# Linear Algebra for Artificial Intelligence

Linear algebra is the mathematical language of modern AI. Every neural network forward pass is a sequence of matrix multiplications. Every image processed by a computer vision model is a matrix of pixel values. Every word embedding is a vector in high-dimensional space. Every recommendation system computes similarities between vectors. Every training step in deep learning uses matrix calculus built on linear algebra foundations.

This notebook builds linear algebra from the ground up — vectors, matrices, operations, transformations, and decompositions — with formulas explained in plain language and implemented in Python using NumPy. The focus is not abstract mathematics for its own sake, but the specific concepts that appear repeatedly when you read about, build, or debug AI systems.

---

## 1. Why Linear Algebra Matters in AI

Machine learning deals with data, and data is represented as numbers arranged in structures. A single measurement is a **scalar** (one number). A list of features for one data point is a **vector**. A table of many data points is a **matrix**. A batch of images or a sequence of time steps is a **tensor** (a generalization to higher dimensions).

When a neural network processes an input, it computes:

$$\mathbf{y} = f(\mathbf{W}\mathbf{x} + \mathbf{b})$$

where $\mathbf{x}$ is an input vector, $\mathbf{W}$ is a weight matrix, $\mathbf{b}$ is a bias vector, and $f$ is an activation function applied element-wise. This single equation — matrix multiplication plus bias plus nonlinearity — is repeated across millions or billions of parameters in models like GPT. Understanding matrix multiplication is understanding the core computation of deep learning.

Linear algebra also provides the tools for dimensionality reduction (PCA via eigenvalues), understanding attention mechanisms (dot products between query and key vectors), measuring similarity (cosine similarity between embeddings), and optimizing models (gradient descent operates on vectors of partial derivatives). Without linear algebra, the internals of AI remain opaque.

In [ ]:
import numpy as np

# Suppress scientific notation for cleaner display
np.set_printoptions(precision=4, suppress=True)

print("NumPy version:", np.__version__)

---

## 2. Scalars, Vectors, Matrices, and Tensors

### 2.1 Scalars

A **scalar** is a single number — a quantity with magnitude but no direction. Examples: a temperature (37.5), a probability (0.85), a loss value (2.34). In Python and NumPy, a scalar is a plain number.

### 2.2 Vectors

A **vector** is an ordered list of numbers. It can represent a point in space, a set of features, or a direction with magnitude. A vector with $n$ components is an element of $\mathbb{R}^n$ (n-dimensional real space).

We write a column vector as:

$$\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix}$$

In machine learning, a vector often represents one data sample. For example, a house described by `[area, bedrooms, age]` is a 3-dimensional vector.

### 2.3 Matrices

A **matrix** is a rectangular array of numbers with $m$ rows and $n$ columns, denoted $\mathbf{A} \in \mathbb{R}^{m \times n}$. The entry in row $i$ and column $j$ is written $a_{ij}$.

$$\mathbf{A} = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \end{bmatrix}$$

In AI, a matrix can represent a dataset (each row is a sample, each column is a feature), a weight layer in a neural network, or a transformation applied to vectors.

### 2.4 Tensors

A **tensor** generalizes scalars, vectors, and matrices to any number of dimensions. A scalar is a rank-0 tensor, a vector is rank-1, a matrix is rank-2. A color image is a rank-3 tensor (height × width × color channels). A batch of images is rank-4 (batch × height × width × channels). Deep learning frameworks like PyTorch and TensorFlow operate primarily on tensors.

In [ ]:
# Scalar
temperature = 37.5
print("Scalar:", temperature, type(temperature))

# Vector (1D array)
house = np.array([1200, 3, 15])  # area sqft, bedrooms, age years
print("\nVector (1D):", house, "shape:", house.shape)

# Column vector (2D array with one column)
col_vector = np.array([[1], [2], [3]])
print("\nColumn vector shape:", col_vector.shape)

# Matrix (2D array)
dataset = np.array([
    [1200, 3, 15],
    [1800, 4, 8],
    [950,  2, 22],
])
print("\nMatrix (dataset, 3 houses x 3 features):")
print(dataset, "shape:", dataset.shape)

# Rank-3 tensor: 2 grayscale images of size 3x3
images = np.zeros((2, 3, 3))
print("\nTensor (2 images x 3 x 3) shape:", images.shape)

---

## 3. Vector Operations

### 3.1 Vector Addition and Scalar Multiplication

Vectors of the same dimension can be added element-wise:

$$\mathbf{a} + \mathbf{b} = \begin{bmatrix} a_1 + b_1 \\ a_2 + b_2 \\ \vdots \\ a_n + b_n \end{bmatrix}$$

A vector can be multiplied by a scalar (each component is scaled):

$$c \cdot \mathbf{v} = \begin{bmatrix} cv_1 \\ cv_2 \\ \vdots \\ cv_n \end{bmatrix}$$

Geometrically, vector addition places vectors tip-to-tail. Scalar multiplication stretches or shrinks the vector (and reverses direction if the scalar is negative).

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("a + b =", a + b)
print("3 * a =", 3 * a)
print("a - b =", a - b)

### 3.2 The Dot Product

The **dot product** (also called inner product or scalar product) of two vectors $\mathbf{a}$ and $\mathbf{b}$ in $\mathbb{R}^n$ is:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \cdots + a_n b_n$$

The result is a **scalar**. The dot product measures how much two vectors point in the same direction. If the dot product is zero, the vectors are **orthogonal** (perpendicular). If positive, they point in similar directions. If negative, they point in opposite directions.

In AI, the dot product is everywhere:
- **Neural networks:** each neuron's output is a dot product of inputs and weights (plus bias).
- **Attention mechanisms:** attention scores are dot products between query and key vectors.
- **Similarity:** related embeddings have high dot products.

A related formula connects the dot product to magnitudes and angles:

$$\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\| \|\mathbf{b}\| \cos\theta$$

where $\|\mathbf{a}\|$ is the magnitude (length) of $\mathbf{a}$ and $\theta$ is the angle between them.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Dot product: multiple methods in NumPy
dot_manual = sum(a[i] * b[i] for i in range(len(a)))
dot_np = np.dot(a, b)
dot_at = a @ b  # @ operator for matrix multiplication / dot product

print("Dot product (manual):", dot_manual)
print("Dot product (np.dot):", dot_np)
print("Dot product (a @ b):", dot_at)

# Orthogonal vectors: dot product = 0
x = np.array([1, 0])
y = np.array([0, 1])
print("\nOrthogonal vectors [1,0] and [0,1], dot =", np.dot(x, y))

### 3.3 Vector Norms (Magnitude)

The **norm** of a vector measures its length. The most common is the **Euclidean norm** (L2 norm):

$$\|\mathbf{v}\|_2 = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2} = \sqrt{\sum_{i=1}^{n} v_i^2}$$

Other norms appear in AI:
- **L1 norm:** $\|\mathbf{v}\|_1 = |v_1| + |v_2| + \cdots + |v_n|$ — used in Lasso regularization.
- **L∞ norm:** $\|\mathbf{v}\|_\infty = \max(|v_1|, |v_2|, \ldots, |v_n|)$ — maximum absolute component.

A **unit vector** has norm 1. Any nonzero vector can be normalized by dividing by its norm: $\hat{\mathbf{v}} = \mathbf{v} / \|\mathbf{v}\|$.

In [ ]:
v = np.array([3, 4])

l2 = np.linalg.norm(v)           # Euclidean norm
l1 = np.linalg.norm(v, ord=1)    # L1 norm
linf = np.linalg.norm(v, ord=np.inf)  # L-infinity norm

print("Vector v =", v)
print("L2 norm (Euclidean):", l2)       # sqrt(9+16) = 5
print("L1 norm:", l1)                   # 3+4 = 7
print("L-infinity norm:", linf)         # max(3,4) = 4

# Unit vector
v_unit = v / l2
print("\nUnit vector:", v_unit)
print("Norm of unit vector:", np.linalg.norm(v_unit))

### 3.4 Cosine Similarity

When comparing embeddings or documents, we often care about **direction** rather than magnitude. **Cosine similarity** measures the cosine of the angle between two vectors:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

Values range from $-1$ (opposite directions) to $1$ (same direction). A value of $0$ means orthogonal. Cosine similarity is widely used in recommendation systems, search engines, and NLP to compare word or sentence embeddings.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Word-like embeddings (simplified 3D example)
king = np.array([1.0, 0.8, 0.2])
queen = np.array([0.9, 0.85, 0.25])
car = np.array([0.1, 0.2, 0.9])

print("cos(king, queen):", round(cosine_similarity(king, queen), 4))
print("cos(king, car):  ", round(cosine_similarity(king, car), 4))
print("\nKing and queen are more similar (higher cosine) than king and car.")

---

## 4. Matrix Operations

### 4.1 Matrix Addition and Scalar Multiplication

Matrices of the same shape are added element-wise, just like vectors. A matrix multiplied by a scalar scales every entry.

### 4.2 Matrix Multiplication

Matrix multiplication is the most important operation in AI. For matrices $\mathbf{A} \in \mathbb{R}^{m \times n}$ and $\mathbf{B} \in \mathbb{R}^{n \times p}$, the product $\mathbf{C} = \mathbf{AB}$ is an $m \times p$ matrix where:

$$c_{ij} = \sum_{k=1}^{n} a_{ik} b_{kj}$$

Entry $c_{ij}$ is the **dot product** of row $i$ of $\mathbf{A}$ and column $j$ of $\mathbf{B}$. The number of columns of $\mathbf{A}$ must equal the number of rows of $\mathbf{B}$.

**Key properties:**
- Matrix multiplication is **not commutative**: $\mathbf{AB} \neq \mathbf{BA}$ in general.
- It is **associative**: $(\mathbf{AB})\mathbf{C} = \mathbf{A}(\mathbf{BC})$.
- Distributive over addition: $\mathbf{A}(\mathbf{B} + \mathbf{C}) = \mathbf{AB} + \mathbf{AC}$.

In a neural network layer, if $\mathbf{X}$ is a batch of inputs (rows = samples), $\mathbf{W}$ is the weight matrix, and $\mathbf{b}$ is the bias:

$$\mathbf{Z} = \mathbf{X}\mathbf{W}^T + \mathbf{b}$$

Each row of $\mathbf{Z}$ is the output of the layer for one sample.

In [ ]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

C = A @ B  # matrix multiplication

print("A =\n", A)
print("B =\n", B)
print("A @ B =\n", C)

# Verify one entry: c_11 = row 0 of A dot col 0 of B = 1*5 + 2*7 = 19
print("\nManual check c[0,0]:", np.dot(A[0], B[:, 0]))

# Non-commutative
print("\nA @ B != B @ A:")
print("A @ B =\n", A @ B)
print("B @ A =\n", B @ A)

### 4.3 Matrix-Vector Multiplication

When a matrix multiplies a vector, the result is a new vector. If $\mathbf{A} \in \mathbb{R}^{m \times n}$ and $\mathbf{x} \in \mathbb{R}^n$, then $\mathbf{y} = \mathbf{Ax} \in \mathbb{R}^m$. Each component $y_i$ is the dot product of row $i$ of $\mathbf{A}$ with $\mathbf{x}$.

Geometrically, matrix-vector multiplication **transforms** the vector — it rotates, scales, and/or shears space. This interpretation is central to understanding how neural network layers reshape data.

In [ ]:
# A single neuron: 3 inputs -> 1 output
weights = np.array([0.5, -0.3, 0.8])   # shape (3,)
inputs = np.array([1.0, 2.0, 3.0])     # shape (3,)
bias = 0.1

# Neuron output = dot(weights, inputs) + bias
output = np.dot(weights, inputs) + bias
print("Single neuron output:", output)

# A layer: 3 inputs -> 2 outputs (2 neurons)
W = np.array([[0.5, -0.3, 0.8],
              [0.2,  0.4, -0.1]])   # shape (2, 3)
b = np.array([0.1, -0.2])           # shape (2,)

layer_output = W @ inputs + b
print("\nLayer output (2 neurons):", layer_output)

### 4.4 Transpose

The **transpose** of a matrix $\mathbf{A}$, written $\mathbf{A}^T$, flips rows and columns: $(\mathbf{A}^T)_{ij} = a_{ji}$. A column vector becomes a row vector and vice versa.

Useful identity: $(\mathbf{AB})^T = \mathbf{B}^T \mathbf{A}^T$

In machine learning, data is often stored with samples as rows and features as columns. Weight matrices are shaped so that multiplying data by weights produces the correct output dimensions.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

print("A shape:", A.shape)
print("A =\n", A)
print("\nA.T shape:", A.T.shape)
print("A.T =\n", A.T)

---

## 5. Linear Combinations and Span

A **linear combination** of vectors $\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k$ is an expression of the form:

$$c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_k \mathbf{v}_k$$

where $c_1, c_2, \ldots, c_k$ are scalars (coefficients). The **span** of a set of vectors is the set of all possible linear combinations of them — all vectors you can "reach" by scaling and adding those vectors.

In a neural network, the output of a layer (before activation) is a linear combination of the input features, with weights as coefficients. The activation function then introduces nonlinearity — without it, stacking layers would still be equivalent to a single linear transformation.

In [ ]:
v1 = np.array([1, 0])
v2 = np.array([0, 1])

# Any point in 2D plane is a linear combination of v1 and v2
c1, c2 = 3, 4
combination = c1 * v1 + c2 * v2
print(f"3 * [1,0] + 4 * [0,1] = {combination}")

# Matrix multiplication IS a linear combination of columns
A = np.array([[1, 2],
              [3, 4]])
x = np.array([5, 6])
result = A @ x  # 5 * col1 + 6 * col2
manual = 5 * A[:, 0] + 6 * A[:, 1]
print(f"\nA @ x = {result}")
print(f"5 * col0 + 6 * col1 = {manual}")

---

## 6. Identity, Inverse, and Determinant

### 6.1 Identity Matrix

The **identity matrix** $\mathbf{I}_n$ is an $n \times n$ matrix with 1s on the diagonal and 0s elsewhere. It acts like the number 1 for multiplication: $\mathbf{AI} = \mathbf{IA} = \mathbf{A}$.

$$\mathbf{I}_3 = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

### 6.2 Matrix Inverse

For a square matrix $\mathbf{A}$, the **inverse** $\mathbf{A}^{-1}$ satisfies:

$$\mathbf{A}\mathbf{A}^{-1} = \mathbf{A}^{-1}\mathbf{A} = \mathbf{I}$$

Not all matrices are invertible. A matrix has an inverse if and only if its **determinant** is nonzero. In AI, inverses appear in solving linear systems, certain optimization formulas, and Mahalanobis distance in anomaly detection.

### 6.3 Determinant

The **determinant** $\det(\mathbf{A})$ is a scalar value that characterizes a square matrix. For a $2 \times 2$ matrix:

$$\det\begin{bmatrix} a & b \\ c & d \end{bmatrix} = ad - bc$$

Geometrically, the absolute value of the determinant is the factor by which the matrix scales area (2D) or volume (3D). A determinant of zero means the transformation collapses space — the matrix is singular (not invertible).

In [ ]:
A = np.array([[4, 7],
              [2, 6]])

det_A = np.linalg.det(A)
A_inv = np.linalg.inv(A)
I_check = A @ A_inv

print("A =\n", A)
print("det(A) =", round(det_A, 4))
print("\nA inverse =\n", np.round(A_inv, 4))
print("\nA @ A_inv (should be identity):\n", np.round(I_check, 4))

# Singular matrix (det = 0, no inverse)
singular = np.array([[1, 2],
                     [2, 4]])
print("\ndet(singular) =", np.linalg.det(singular))

---

## 7. Solving Systems of Linear Equations

A system of linear equations can be written in matrix form:

$$\mathbf{Ax} = \mathbf{b}$$

where $\mathbf{A}$ is the coefficient matrix, $\mathbf{x}$ is the vector of unknowns, and $\mathbf{b}$ is the constants vector. If $\mathbf{A}$ is invertible, the solution is:

$$\mathbf{x} = \mathbf{A}^{-1}\mathbf{b}$$

In practice, NumPy uses more numerically stable methods (LU decomposition) rather than computing the inverse directly. Linear regression's normal equation $\mathbf{w} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ is a direct application.

In [ ]:
# Solve: 2x + y = 5,  x + 3y = 7
A = np.array([[2, 1],
              [1, 3]])
b = np.array([5, 7])

x = np.linalg.solve(A, b)
print("Solution x, y =", x)
print("Verification A @ x =", A @ x)

---

## 8. Eigenvalues and Eigenvectors

For a square matrix $\mathbf{A}$, an **eigenvector** $\mathbf{v}$ is a nonzero vector that, when $\mathbf{A}$ is applied, only scales (does not change direction):

$$\mathbf{Av} = \lambda \mathbf{v}$$

The scalar $\lambda$ is the corresponding **eigenvalue**. Intuitively, eigenvectors are the special directions that a linear transformation acts on by pure stretching or flipping.

Eigenvalues and eigenvectors are fundamental in AI:
- **Principal Component Analysis (PCA)** finds eigenvectors of the covariance matrix to identify directions of maximum variance in data.
- **PageRank** (Google's original algorithm) uses the dominant eigenvector of a link matrix.
- **Stability analysis** of neural network training examines eigenvalues of the Hessian matrix.
- **Spectral clustering** uses eigenvectors of graph Laplacians.

In [ ]:
A = np.array([[4, 1],
              [2, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print("Matrix A =\n", A)
print("\nEigenvalues:", np.round(eigenvalues, 4))
print("\nEigenvectors (columns):")
print(np.round(eigenvectors, 4))

# Verify: A @ v = lambda * v for first eigenvector
v = eigenvectors[:, 0]
lam = eigenvalues[0]
print("\nVerify A @ v = lambda * v:")
print("A @ v =     ", np.round(A @ v, 4))
print("lambda * v =", np.round(lam * v, 4))

---

## 9. Orthogonality and Projections

Two vectors are **orthogonal** if their dot product is zero: $\mathbf{u} \cdot \mathbf{v} = 0$. They form a right angle. A set of vectors is **orthonormal** if each has unit length and every pair is orthogonal.

The **projection** of vector $\mathbf{b}$ onto vector $\mathbf{a}$ is:

$$\text{proj}_{\mathbf{a}} \mathbf{b} = \frac{\mathbf{a} \cdot \mathbf{b}}{\mathbf{a} \cdot \mathbf{a}} \mathbf{a}$$

Projection finds the closest point on a line (or subspace) to a given point. In machine learning, gradient descent updates are related to projecting the negative gradient onto search directions. Orthogonal weight initialization helps training stability in deep networks.

In [ ]:
a = np.array([3, 0])  # x-axis direction
b = np.array([2, 3])  # point to project

proj = (np.dot(a, b) / np.dot(a, a)) * a
print("Project b onto a:", proj)
print("Residual (orthogonal component):", b - proj)
print("Dot product of residual and a (should be ~0):", np.dot(b - proj, a))

---

## 10. Singular Value Decomposition (SVD)

The **Singular Value Decomposition** factorizes any matrix $\mathbf{A}$ (not necessarily square) into three matrices:

$$\mathbf{A} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$$

- $\mathbf{U}$: orthogonal matrix (left singular vectors)
- $\mathbf{\Sigma}$: diagonal matrix of **singular values** (nonnegative, sorted descending)
- $\mathbf{V}^T$: transpose of orthogonal matrix (right singular vectors)

SVD is one of the most important decompositions in applied mathematics and AI. Applications include:
- **Dimensionality reduction:** keeping only the largest singular values approximates the matrix with lower rank.
- **Image compression:** represent images with fewer components.
- **Recommendation systems:** matrix factorization methods (like latent factor models) are related to SVD.
- **Noise reduction:** small singular values often correspond to noise.
- **Pseudoinverse:** solving least-squares problems when $\mathbf{A}$ is not square or invertible.

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])

U, S, Vt = np.linalg.svd(A, full_matrices=False)

print("A shape:", A.shape)
print("Singular values:", np.round(S, 4))
print("\nU shape:", U.shape)
print("S (diagonal):", np.round(S, 4))
print("Vt shape:", Vt.shape)

# Reconstruct A from SVD
Sigma = np.diag(S)
A_reconstructed = U @ Sigma @ Vt
print("\nReconstructed A:\n", np.round(A_reconstructed, 4))

### 10.1 Low-Rank Approximation (Dimensionality Reduction)

By keeping only the top $k$ singular values and corresponding vectors, we obtain the best rank-$k$ approximation of the matrix. This is the idea behind **Latent Semantic Analysis** in NLP and many compression techniques.

In [ ]:
# Simple example: compress a matrix keeping top 1 singular value
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10, 11, 12]], dtype=float)

U, S, Vt = np.linalg.svd(A, full_matrices=False)

k = 1  # keep only 1 component
A_approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

print("Original matrix (4x3):")
print(A)
print(f"\nRank-{k} approximation:")
print(np.round(A_approx, 2))
print(f"\nReconstruction error (Frobenius norm): {np.linalg.norm(A - A_approx, 'fro'):.4f}")

---

## 11. Principal Component Analysis (PCA) — Application

**PCA** reduces the dimensionality of data while preserving as much variance as possible. It finds the directions (principal components) along which data varies most, which are the eigenvectors of the **covariance matrix**.

Steps:
1. Center the data (subtract the mean).
2. Compute the covariance matrix.
3. Find eigenvectors and eigenvalues of the covariance matrix.
4. Project data onto the top $k$ eigenvectors.

PCA is used for visualization (reducing to 2D or 3D), noise reduction, feature extraction, and speeding up machine learning by removing redundant dimensions.

In [ ]:
# Generate 2D data with correlation
np.random.seed(42)
n = 100
x1 = np.random.randn(n) * 2
x2 = x1 * 0.8 + np.random.randn(n) * 0.5
data = np.column_stack([x1, x2])

# Step 1: Center
mean = data.mean(axis=0)
centered = data - mean

# Step 2: Covariance matrix
cov = np.cov(centered, rowvar=False)
print("Covariance matrix:\n", np.round(cov, 4))

# Step 3: Eigen decomposition
eigenvalues, eigenvectors = np.linalg.eig(cov)
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("\nEigenvalues (variance along each PC):", np.round(eigenvalues, 4))
print("First principal component direction:", np.round(eigenvectors[:, 0], 4))

# Step 4: Project to 1D
projected = centered @ eigenvectors[:, 0]
print(f"\nReduced from 2D to 1D. Projected shape: {projected.shape}")
print("Variance explained by PC1: {:.1f}%".format(100 * eigenvalues[0] / eigenvalues.sum()))

---

## 12. Building a Simple Neural Network Layer with Linear Algebra

A fully connected neural network layer applies: $\mathbf{Z} = \mathbf{X}\mathbf{W}^T + \mathbf{b}$, then an activation function element-wise. The code below implements this for a batch of samples.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Batch of 4 samples, 3 features each
X = np.array([
    [1.0, 0.5, -0.2],
    [0.3, 1.0,  0.8],
    [-0.5, 0.2, 1.5],
    [2.0, -1.0, 0.0],
])

# Layer: 3 inputs -> 2 hidden units
W = np.array([[0.5, -0.3, 0.8],
              [0.2,  0.4, -0.1]])
b = np.array([0.1, -0.2])

# Forward pass
Z = X @ W.T + b   # linear transformation
H = relu(Z)       # activation

print("Input batch X (4 samples x 3 features):")
print(np.round(X, 3))
print("\nWeight matrix W (2 neurons x 3 inputs):")
print(W)
print("\nLinear output Z = X @ W.T + b:")
print(np.round(Z, 4))
print("\nAfter ReLU activation:")
print(np.round(H, 4))

# Output layer: 2 hidden -> 1 output (binary classification)
W2 = np.array([[0.6, -0.4]])
b2 = np.array([-0.1])
output = sigmoid(H @ W2.T + b2)
print("\nFinal probabilities (sigmoid output):")
print(np.round(output.flatten(), 4))

Every step above is matrix multiplication and addition. A deep network with 100 layers is 100 consecutive matrix multiplications (with nonlinearities between). GPT-scale models perform these operations with matrices containing billions of parameters — but the underlying math is identical to what we just implemented.

---

## 13. Distance Metrics in Vector Space

Comparing data points in feature space requires **distance metrics**. Common ones in AI:

**Euclidean distance** between $\mathbf{a}$ and $\mathbf{b}$:

$$d(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|_2 = \sqrt{\sum_{i=1}^{n}(a_i - b_i)^2}$$

**Manhattan distance** (L1):

$$d(\mathbf{a}, \mathbf{b}) = \sum_{i=1}^{n} |a_i - b_i|$$

**Cosine distance** $= 1 - \cos(\theta)$ — used when direction matters more than magnitude.

K-nearest neighbors (KNN), clustering (K-means), and anomaly detection all rely on distance metrics to group or flag data points.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

euclidean = np.linalg.norm(a - b)
manhattan = np.sum(np.abs(a - b))
cosine_dist = 1 - cosine_similarity(a, b)

print("Euclidean distance:", round(euclidean, 4))
print("Manhattan distance: ", manhattan)
print("Cosine distance:    ", round(cosine_dist, 4))

---

## 14. Key Formulas Reference

| Concept | Formula |
|---------|--------|
| Dot product | $\mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i$ |
| L2 norm | $\|\mathbf{v}\|_2 = \sqrt{\sum_i v_i^2}$ |
| Cosine similarity | $\cos\theta = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$ |
| Matrix multiply | $c_{ij} = \sum_k a_{ik} b_{kj}$ |
| 2×2 determinant | $\det = ad - bc$ |
| Inverse | $\mathbf{AA}^{-1} = \mathbf{I}$ |
| Eigenvalue | $\mathbf{Av} = \lambda\mathbf{v}$ |
| SVD | $\mathbf{A} = \mathbf{U}\mathbf{\Sigma}\mathbf{V}^T$ |
| Projection | $\text{proj}_{\mathbf{a}}\mathbf{b} = \frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{a}\cdot\mathbf{a}}\mathbf{a}$ |
| Neural layer | $\mathbf{Z} = \mathbf{XW}^T + \mathbf{b}$ |

---

## 15. Summary

Linear algebra provides the structural foundation of artificial intelligence. **Vectors** represent data points and embeddings. **Matrices** represent datasets, weight layers, and linear transformations. **Matrix multiplication** is the core computation in neural networks. The **dot product** measures similarity and computes neuron outputs. **Eigenvalues and eigenvectors** power PCA and many spectral methods. **SVD** enables dimensionality reduction, compression, and matrix factorization.

Every concept in this notebook — from basic vector addition to SVD-based approximation — appears in production AI systems. NumPy makes these operations practical in Python; deep learning frameworks extend them to GPUs and tensors at massive scale. Mastering linear algebra transforms AI from a black box into a system you can reason about, debug, and build.